# CNN-LSTM 量化选股策略

**论文参考**: Huishan Zhuang (2022), *Research on Quantitative Stock Selection Strategy Based on CNN-LSTM*

本notebook实现 Conv1D + MaxPool + LSTM 混合架构，CNN提取局部模式特征后送入LSTM建模时序依赖，预测贵州茅台(600519)次日涨跌方向。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### CNN-LSTM 混合架构

**Stage 1: 1D卷积特征提取**

Conv1D沿时间轴滑动，提取局部模式:
$$h_i^{(l)} = \text{ReLU}\left(\sum_{k=0}^{K-1} W_k^{(l)} \cdot x_{i+k} + b^{(l)}\right)$$

其中 $K$ 为卷积核大小, $l$ 为滤波器索引。

**MaxPooling**: 降维并保留最显著特征
$$p_i = \max(h_{2i}, h_{2i+1})$$

**Stage 2: LSTM 时序建模**

CNN输出送入LSTM捕捉长期依赖:
$$h_t = \text{LSTM}(p_t, h_{t-1}, C_{t-1})$$

**全连接分类**:
$$\hat{y} = \sigma(W_{fc} \cdot h_T + b_{fc})$$

### 损失函数
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

### 架构参数
- Conv1D: filters=32, kernel_size=3, padding=1
- MaxPool1D: kernel_size=2
- LSTM: hidden_size=64
- FC: 64 -> 1

In [ ]:
# ===== Cell 3: CNN滤波器滑动动画 =====
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
T = 30  # 序列长度
t = np.arange(T)
signal = np.sin(0.3 * t) + 0.3 * np.random.randn(T)  # 模拟价格序列
kernel_size = 3

# 模拟一个学到的卷积核 (边缘检测器)
kernel = np.array([-1.0, 0.0, 1.0])

# 计算卷积输出
conv_output = np.convolve(signal, kernel, mode='valid')

frames = []
for pos in range(T - kernel_size + 1):
    # 输入信号
    highlight_x = t[pos:pos+kernel_size]
    highlight_y = signal[pos:pos+kernel_size]
    
    frames.append(go.Frame(
        data=[
            go.Scatter(x=t, y=signal, mode='lines', name='输入序列',
                       line=dict(color='#333', width=1.5)),
            go.Scatter(x=highlight_x, y=highlight_y, mode='markers+lines',
                       name='卷积窗口', line=dict(color='#F44336', width=3),
                       marker=dict(size=10, color='#F44336')),
            go.Bar(x=list(range(pos+1)), y=list(conv_output[:pos+1]),
                   name='卷积输出', marker_color='#2196F3', opacity=0.7),
        ],
        name=f'位置 {pos}'
    ))

fig = make_subplots(rows=2, cols=1, subplot_titles=['CNN 1D卷积滑动过程', '卷积输出 → LSTM输入'],
                    row_heights=[0.5, 0.5])

# 初始帧
fig.add_trace(go.Scatter(x=t, y=signal, mode='lines', name='输入序列',
                         line=dict(color='#333')), row=1, col=1)
fig.add_trace(go.Scatter(x=[], y=[], mode='markers+lines', name='卷积窗口',
                         line=dict(color='#F44336', width=3)), row=1, col=1)
fig.add_trace(go.Bar(x=[], y=[], name='卷积输出', marker_color='#2196F3'), row=2, col=1)

fig.update_layout(
    title='Conv1D 滤波器滑动过程 (kernel=[-1, 0, 1], 边缘检测)',
    height=500, width=900,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='\u25b6 播放', method='animate',
             args=[None, {'frame': {'duration': 150}, 'transition': {'duration': 50}}]),
        dict(label='\u23f8 暂停', method='animate',
             args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                    label=f.name, method='animate') for f in frames],
    )],
)
fig.frames = frames
fig.show()

In [ ]:
# ===== Cell 4: Imports & Setup =====
import sys
sys.path.insert(0, '../shared')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from data_fetcher import get_stock_daily
from factors import rsi, macd, bollinger_bands, volatility
from backtest_engine import Backtester
from visualization import plot_equity_curve, plot_drawdown, plot_metrics_table
from constants import STOCK_MOUTAI, INITIAL_CAPITAL

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ===== Cell 5: 数据获取 =====
df = get_stock_daily(STOCK_MOUTAI, start_date="20200101", end_date="20241231")
print(f"数据形状: {df.shape}, 日期范围: {df.index[0]} ~ {df.index[-1]}")
df.head()

In [ ]:
# ===== Cell 6: 特征工程 & 滑动窗口 =====
df['close_pct'] = df['close'].pct_change()
df['volume_pct'] = df['volume'].pct_change()
df['rsi'] = rsi(df['close'], window=14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb_df = bollinger_bands(df['close'])
df['bb_pctb'] = bb_df['bb_pctb']
vol_df = volatility(df['close'], windows=[20])
df['volatility'] = vol_df['vol_20']
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)

feature_cols = ['close_pct', 'volume_pct', 'rsi', 'macd_hist', 'bb_pctb', 'volatility']
df = df.dropna(subset=feature_cols + ['target'])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

train_mask = df.index < '2024-01-01'
test_mask = df.index >= '2024-01-01'
train_data = df[train_mask].copy()
test_data = df[test_mask].copy()

scaler.fit(train_data[feature_cols])
train_data[feature_cols] = scaler.transform(train_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

WINDOW = 30

def create_sequences(data, feature_cols, target_col, window):
    X, y, dates = [], [], []
    features = data[feature_cols].values
    targets = data[target_col].values
    idx = data.index
    for i in range(window, len(data)):
        X.append(features[i-window:i])
        y.append(targets[i])
        dates.append(idx[i])
    return np.array(X), np.array(y), dates

X_train, y_train, dates_train = create_sequences(train_data, feature_cols, 'target', WINDOW)
X_test, y_test, dates_test = create_sequences(test_data, feature_cols, 'target', WINDOW)

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"测试集: X={X_test.shape}, y={y_test.shape}")

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
                          batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test)),
                         batch_size=32, shuffle=False)

In [ ]:
# ===== Cell 7: CNN-LSTM 模型定义 & 训练 =====

class CNNLSTM(nn.Module):
    """Conv1D(32, kernel=3) -> MaxPool(2) -> LSTM(64) -> FC(1)"""
    def __init__(self, input_size=6, cnn_filters=32, kernel_size=3,
                 lstm_hidden=64, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, cnn_filters, kernel_size, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout1 = nn.Dropout(dropout)
        
        # Conv1d输出: (B, cnn_filters, T//2)
        # LSTM输入需要: (B, T//2, cnn_filters)
        self.lstm = nn.LSTM(cnn_filters, lstm_hidden, num_layers=1, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        # x: (B, T, input_size)
        x = x.permute(0, 2, 1)   # (B, input_size, T) for Conv1d
        x = self.relu(self.conv1(x))   # (B, cnn_filters, T)
        x = self.pool(x)               # (B, cnn_filters, T//2)
        x = self.dropout1(x)
        x = x.permute(0, 2, 1)   # (B, T//2, cnn_filters) for LSTM
        out, _ = self.lstm(x)     # (B, T//2, lstm_hidden)
        out = self.dropout2(out[:, -1, :])  # 取最后时间步
        return torch.sigmoid(self.fc(out)).squeeze(-1)


model = CNNLSTM(input_size=6, cnn_filters=32, kernel_size=3,
                lstm_hidden=64, dropout=0.2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

history = []
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    history.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50, Loss: {avg_loss:.4f}")

# 预测
model.eval()
preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        pred = model(X_batch.to(device))
        preds.extend(pred.cpu().numpy())
preds = np.array(preds)

acc = ((preds > 0.5).astype(int) == y_test).mean()
print(f"\n测试集准确率: {acc:.4f}")

In [ ]:
# ===== Cell 8: 信号生成 & 回测 =====
test_prices = df.loc[dates_test, 'close']
signals = pd.Series((preds > 0.5).astype(int), index=dates_test)

bt = Backtester(initial_capital=INITIAL_CAPITAL)
result = bt.run(test_prices, signals)

print("CNN-LSTM 回测结果:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# ===== Cell 9: 可视化 =====

# 1. 训练损失
fig = go.Figure()
fig.add_trace(go.Scatter(y=history, mode='lines', name='训练损失',
                         line=dict(color='#FF9800', width=2)))
fig.update_layout(title='CNN-LSTM 训练损失', xaxis_title='Epoch',
                  yaxis_title='BCE Loss', height=350, width=700)
fig.show()

# 2. 预测概率 vs 实际
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['预测概率 vs 阈值', '收盘价与信号'])
fig.add_trace(go.Scatter(x=dates_test, y=preds, mode='lines',
                         name='预测概率', line=dict(color='#FF9800')), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', row=1, col=1)

buy_mask = signals.diff().fillna(0) > 0
sell_mask = signals.diff().fillna(0) < 0
fig.add_trace(go.Scatter(x=test_prices.index, y=test_prices.values,
                         mode='lines', name='收盘价', line=dict(color='#333')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[buy_mask], y=test_prices[buy_mask].values,
                         mode='markers', name='买入',
                         marker=dict(color='green', size=8, symbol='triangle-up')), row=2, col=1)
fig.add_trace(go.Scatter(x=test_prices.index[sell_mask], y=test_prices[sell_mask].values,
                         mode='markers', name='卖出',
                         marker=dict(color='red', size=8, symbol='triangle-down')), row=2, col=1)
fig.update_layout(height=600, width=1000, title='CNN-LSTM 预测与交易信号')
fig.show()

# 3. 权益曲线
plot_equity_curve(result['equity_curve'], title='CNN-LSTM 策略收益曲线')

# 4. 回撤
plot_drawdown(result['equity_curve'], title='CNN-LSTM 策略回撤')

# 5. 绩效表
plot_metrics_table(result['metrics'], title='CNN-LSTM 绩效指标')

## 结果讨论

### CNN-LSTM 架构分析
1. **Conv1D**: 3-day卷积核能捕捉短期价格模式(如三日反转、跳空缺口)
2. **MaxPool**: 2倍下采样降低LSTM的序列长度，减少计算量
3. **LSTM**: 在CNN特征上建模中期依赖关系

### 与纯LSTM对比
- CNN预处理减少了噪声，LSTM能更好地聚焦于有意义的模式
- 参数量适中，训练收敛较快
- 在趋势行情中表现优于震荡行情

### 改进方向
- 多尺度卷积核(3, 5, 7)并行提取不同周期的模式
- 加入残差连接缓解深层网络梯度问题
- Dilated Conv1D 扩大感受野

### 参考
- Zhuang, H. (2022). Research on Quantitative Stock Selection Strategy Based on CNN-LSTM.